# Dental Appointment Slot Detection Using CNN

---

## Section 1 — Project Introduction

### Goal
Dental clinics often use grid-based scheduling software where each column represents a practitioner and each row represents a time slot. **Yellow cells** indicate free/available slots, while cells containing **text, colored blocks (green, pink, gray), or appointment details** represent booked slots.

Our goal is to **automatically detect which appointment slots are available vs. booked** from a screenshot of such a scheduling interface.

### Why Rule-Based Image Processing Sometimes Fails
A pure rule-based approach (e.g., thresholding on yellow pixel ratio) can break when:
- The shade of yellow varies across the image or between software versions.
- "Booked" slots use a wide range of colors (green, pink, purple, gray).
- Text overlays, grid lines, and UI artifacts create false positives.
- Image quality, resolution, or compression artifacts differ between captures.

### Why a CNN Classifier Can Help
A **Convolutional Neural Network** learns visual features directly from labeled examples. Instead of hand-tuning color thresholds, we:
1. Extract individual slot cell images from the scheduling grid.
2. Manually label a small set as **free** or **booked**.
3. Train a lightweight CNN to generalize the distinction.

This approach is more **robust to visual variation** and can be retrained easily when the scheduling software changes its theme.

---
## Section 2 — Install & Import Libraries

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install opencv-python numpy matplotlib tensorflow pytesseract Pillow

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import pytesseract
import os
import glob
from sklearn.model_selection import train_test_split

print(f"OpenCV version:      {cv2.__version__}")
print(f"NumPy version:       {np.__version__}")
print(f"TensorFlow version:  {tf.__version__}")
print(f"Pytesseract path:    {pytesseract.pytesseract.tesseract_cmd}")

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

---
## Section 3 — Load and Visualize the Scheduling Image

We load the dental scheduling screenshot and convert it from OpenCV's default **BGR** color space to **RGB** for correct display in matplotlib.

In [ ]:
IMAGE_PATH = "./schedule.png"

image_bgr = cv2.imread(IMAGE_PATH)
if image_bgr is None:
    raise FileNotFoundError(f"Cannot open image: {IMAGE_PATH}")

image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

print(f"Image shape: {image_bgr.shape}  (height x width x channels)")
print(f"Image dtype: {image_bgr.dtype}")

plt.figure(figsize=(16, 8))
plt.imshow(image_rgb)
plt.title("Dental Scheduling Grid — Original Image", fontsize=14)
plt.axis("off")
plt.tight_layout()
plt.show()

---
## Section 4 — Image Preprocessing

We apply a sequence of preprocessing steps to prepare the image for slot extraction:

1. **Grayscale conversion** — reduce 3-channel color to single intensity channel.
2. **Gaussian blur** — reduce high-frequency noise.
3. **Otsu thresholding** — automatically find an optimal binarization threshold.
4. **Morphological opening** — remove small noise specks.

Each intermediate result is displayed and saved to disk.

In [ ]:
os.makedirs("preprocessing_outputs", exist_ok=True)

# Step 4.1 — Grayscale
gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
cv2.imwrite("preprocessing_outputs/01_grayscale.png", gray)

plt.figure(figsize=(14, 6))
plt.imshow(gray, cmap="gray")
plt.title("Step 4.1 — Grayscale")
plt.axis("off")
plt.show()
print(f"Saved: preprocessing_outputs/01_grayscale.png  |  Shape: {gray.shape}")

In [ ]:
# Step 4.2 — Gaussian Blur
blurred = cv2.GaussianBlur(gray, (3, 3), 0)
cv2.imwrite("preprocessing_outputs/02_gaussian_blur.png", blurred)

plt.figure(figsize=(14, 6))
plt.imshow(blurred, cmap="gray")
plt.title("Step 4.2 — Gaussian Blur (3×3)")
plt.axis("off")
plt.show()
print(f"Saved: preprocessing_outputs/02_gaussian_blur.png")

In [ ]:
# Step 4.3 — Otsu Thresholding
thresh_val, binary = cv2.threshold(
    blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
)
cv2.imwrite("preprocessing_outputs/03_otsu_threshold.png", binary)

plt.figure(figsize=(14, 6))
plt.imshow(binary, cmap="gray")
plt.title(f"Step 4.3 — Otsu Thresholding (auto threshold = {thresh_val:.0f})")
plt.axis("off")
plt.show()
print(f"Saved: preprocessing_outputs/03_otsu_threshold.png")

In [ ]:
# Step 4.4 — Morphological Opening (noise removal)
kernel = np.ones((2, 2), np.uint8)
cleaned = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
cv2.imwrite("preprocessing_outputs/04_morph_opening.png", cleaned)

plt.figure(figsize=(14, 6))
plt.imshow(cleaned, cmap="gray")
plt.title("Step 4.4 — Morphological Opening (2×2 kernel)")
plt.axis("off")
plt.show()
print(f"Saved: preprocessing_outputs/04_morph_opening.png")

---
## Section 5 — Extract Slot Cells from the Grid

We need to isolate individual time-slot cells from the scheduling grid. Our approach:

1. **Detect the green-bordered grid area** using HSV color filtering.
2. **Find column boundaries** by locating vertical separator lines.
3. **Divide each column into hourly rows** based on the grid's time structure.
4. **Crop each cell** and save it for labeling.

The cropped images are saved into `dataset/free/` and `dataset/booked/` directories. **After running this cell, you must manually move misclassified images** between folders to correct the labels. The auto-labeling here is a starting heuristic based on yellow pixel ratio.

In [ ]:
def group_consecutive(arr, gap=3):
    """Group sorted integers that are within `gap` of each other."""
    if len(arr) == 0:
        return []
    groups, current = [], [arr[0]]
    for v in arr[1:]:
        if v - current[-1] <= gap:
            current.append(v)
        else:
            groups.append(current)
            current = [v]
    groups.append(current)
    return groups


def find_grid_area(image):
    """Detect the green-bordered scheduling grid. Returns (x_left, x_right, y_top, y_bottom)."""
    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)
    h, s, v = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]
    green = (h >= 35) & (h <= 85) & (s > 100) & (v > 100)

    height, width = image.shape[:2]

    green_per_col = np.sum(green, axis=0)
    high_cols = np.where(green_per_col > height * 0.5)[0]
    col_groups = group_consecutive(high_cols)
    if len(col_groups) >= 2:
        x_left = col_groups[0][-1] + 1
        x_right = col_groups[-1][0] - 1
    else:
        x_left, x_right = 30, width - 30

    green_in_grid = np.sum(green[:, x_left:x_right], axis=1)
    grid_width = max(x_right - x_left, 1)
    high_rows = np.where(green_in_grid > grid_width * 0.8)[0]
    row_groups = group_consecutive(high_rows)

    if len(row_groups) >= 3:
        y_top = row_groups[-2][-1] + 1
        y_bottom = row_groups[-1][0] - 1
    elif len(row_groups) >= 2:
        y_top = row_groups[0][-1] + 1
        y_bottom = row_groups[-1][0] - 1
    else:
        y_top, y_bottom = 70, height - 40

    return x_left, x_right, y_top, y_bottom


def find_columns(image, x_left, x_right, y_top, y_bottom):
    """Find column boundaries by detecting vertical separator lines."""
    gray_img = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    region = gray_img[y_top:y_bottom, x_left:x_right]

    dark_count = np.sum(region < 80, axis=0)
    threshold = (y_bottom - y_top) * 0.15
    dark_cols = np.where(dark_count > threshold)[0]

    groups = group_consecutive(dark_cols)
    separators = [int(np.mean(g)) + x_left for g in groups]

    edges = [x_left] + separators + [x_right]
    bounds = [(edges[i], edges[i + 1]) for i in range(len(edges) - 1)]
    return [(a, b) for a, b in bounds if b - a >= 20]


x_left, x_right, y_top, y_bottom = find_grid_area(image_bgr)
print(f"Grid area: x=[{x_left}, {x_right}], y=[{y_top}, {y_bottom}]")

all_cols = find_columns(image_bgr, x_left, x_right, y_top, y_bottom)
print(f"Detected {len(all_cols)} columns")

# Separate the narrow time-label column from practitioner columns
widths = [b - a for a, b in all_cols]
median_w = sorted(widths)[len(widths) // 2]
time_col = None
data_cols = []
for bounds in all_cols:
    if bounds[1] - bounds[0] < median_w * 0.6 and time_col is None:
        time_col = bounds
    else:
        data_cols.append(bounds)
if time_col is None:
    time_col = (x_left, all_cols[0][0])

header_h = max(18, int((y_bottom - y_top) * 0.03))
header_y_top = y_top
header_y_bot = y_top + header_h
content_y_top = header_y_bot + 2

print(f"Time-label column: x=[{time_col[0]}, {time_col[1]}]")
print(f"Practitioner columns: {len(data_cols)}")

# Visualize detected grid area and columns
vis = image_rgb.copy()
cv2.rectangle(vis, (x_left, y_top), (x_right, y_bottom), (255, 0, 0), 2)
for (cx0, cx1) in all_cols:
    cv2.line(vis, (cx0, y_top), (cx0, y_bottom), (0, 0, 255), 1)
    cv2.line(vis, (cx1, y_top), (cx1, y_bottom), (0, 0, 255), 1)

plt.figure(figsize=(16, 8))
plt.imshow(vis)
plt.title(f"Detected Grid (blue) and {len(all_cols)} Column Boundaries (red)", fontsize=13)
plt.axis("off")
plt.tight_layout()
plt.show()

### 5.1 — Identify Dentist Names via OCR

Each column in the scheduling grid corresponds to a **dentist/practitioner**. We use **pytesseract** to OCR the header row and extract their names. This mapping is critical — it allows the user to query availability by dentist name.

In [ ]:
def extract_column_headers(image, col_bounds, y_start, y_end):
    """OCR the header cell above each practitioner column."""
    if y_end <= y_start or y_end - y_start < 5:
        return [""] * len(col_bounds)

    names = []
    for x0, x1 in col_bounds:
        margin = max(2, int((x1 - x0) * 0.04))
        cell = image[y_start:y_end, x0 + margin:x1 - margin]
        if cell.size == 0:
            names.append("")
            continue
        gray_cell = cv2.cvtColor(cell, cv2.COLOR_BGR2GRAY)
        scaled = cv2.resize(gray_cell, None, fx=4, fy=4, interpolation=cv2.INTER_CUBIC)
        _, bin_cell = cv2.threshold(scaled, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        text = pytesseract.image_to_string(bin_cell, config="--psm 7").strip()
        text = text.strip("|-_.,;:!?")
        names.append(text if len(text) > 1 else "")
    return names


dentist_names = extract_column_headers(image_bgr, data_cols, header_y_top, header_y_bot)

print("Detected Dentist / Practitioner Names:")
print("-" * 50)
for i, name in enumerate(dentist_names):
    cx0, cx1 = data_cols[i]
    display_name = name if name else "(unrecognized)"
    print(f"  Column {i+1:2d}  |  x=[{cx0:4d}, {cx1:4d}]  |  {display_name}")

# Visualize header crops
n_show = min(len(data_cols), 10)
fig, axes = plt.subplots(1, n_show, figsize=(min(20, n_show * 2.5), 2))
if n_show == 1:
    axes = [axes]
for i in range(n_show):
    x0, x1 = data_cols[i]
    header_crop = image_rgb[header_y_top:header_y_bot, x0:x1]
    axes[i].imshow(header_crop)
    label = dentist_names[i] if dentist_names[i] else f"Col {i+1}"
    axes[i].set_title(label, fontsize=8, rotation=15)
    axes[i].axis("off")
plt.suptitle("Practitioner Header Cells (OCR'd)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
def estimate_row_height(y_top, y_bottom, expected_hours=13):
    """Estimate pixel height per hour-slot based on grid span (typically 7am-20pm)."""
    total_height = y_bottom - y_top
    return total_height // expected_hours


def is_yellow_cell(cell_bgr, yellow_ratio_threshold=0.40):
    """Heuristic: a cell is 'free' if a large fraction of its pixels are yellow."""
    hsv = cv2.cvtColor(cell_bgr, cv2.COLOR_BGR2HSV)
    h, s, v = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]
    yellow_mask = (h >= 18) & (h <= 38) & (s > 60) & (v > 180)
    ratio = np.sum(yellow_mask) / max(cell_bgr.shape[0] * cell_bgr.shape[1], 1)
    return ratio > yellow_ratio_threshold


row_h = estimate_row_height(content_y_top, y_bottom)
print(f"Estimated row height: {row_h} px")
print(f"Data columns (practitioners): {len(data_cols)}")

# Create dataset directories
os.makedirs("dataset/free", exist_ok=True)
os.makedirs("dataset/booked", exist_ok=True)

slot_id = 0
free_count = 0
booked_count = 0

for col_idx, (cx0, cx1) in enumerate(data_cols):
    margin = 3
    y = content_y_top
    row_idx = 0
    while y + row_h <= y_bottom:
        cell = image_bgr[y:y + row_h, cx0 + margin:cx1 - margin]
        if cell.size == 0:
            y += row_h
            row_idx += 1
            continue

        label = "free" if is_yellow_cell(cell) else "booked"
        fname = f"col{col_idx:02d}_row{row_idx:02d}_{slot_id:04d}.png"
        cv2.imwrite(f"dataset/{label}/{fname}", cell)

        if label == "free":
            free_count += 1
        else:
            booked_count += 1

        slot_id += 1
        y += row_h
        row_idx += 1

print(f"\nTotal slots extracted: {slot_id}")
print(f"  Auto-labeled FREE:   {free_count}")
print(f"  Auto-labeled BOOKED: {booked_count}")
print(f"\nFiles saved to: dataset/free/ and dataset/booked/")
print("Review and manually correct labels before training!")

In [ ]:
# Display a sample of extracted slot images
free_samples = sorted(glob.glob("dataset/free/*.png"))[:6]
booked_samples = sorted(glob.glob("dataset/booked/*.png"))[:6]

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle("Sample Extracted Slots", fontsize=14)

for i, path in enumerate(free_samples):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    axes[0, i].imshow(img)
    axes[0, i].set_title(f"FREE", fontsize=10, color="green")
    axes[0, i].axis("off")

for i, path in enumerate(booked_samples):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    axes[1, i].imshow(img)
    axes[1, i].set_title(f"BOOKED", fontsize=10, color="red")
    axes[1, i].axis("off")

# Hide unused subplots
for row in axes:
    for ax in row:
        if not ax.images:
            ax.set_visible(False)

plt.tight_layout()
plt.show()

---
## Section 6 — Prepare Dataset

We now build a proper training dataset from the labeled images:

1. Read all images from `dataset/free/` (label 0) and `dataset/booked/` (label 1).
2. Resize each image to **64×64** pixels for uniform CNN input.
3. Normalize pixel values to **[0, 1]**.
4. One-hot encode the labels.
5. Split into **training (80%)** and **validation (20%)** sets.

In [ ]:
IMG_SIZE = 64
CLASS_NAMES = ["free", "booked"]


def load_dataset(base_dir="dataset", img_size=IMG_SIZE):
    """Load images and labels from dataset/free and dataset/booked."""
    images = []
    labels = []

    for label_idx, class_name in enumerate(CLASS_NAMES):
        class_dir = os.path.join(base_dir, class_name)
        if not os.path.isdir(class_dir):
            print(f"Warning: {class_dir} not found, skipping.")
            continue

        paths = glob.glob(os.path.join(class_dir, "*.png"))
        for path in paths:
            img = cv2.imread(path)
            if img is None:
                continue
            img = cv2.resize(img, (img_size, img_size))
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            images.append(img)
            labels.append(label_idx)

    images = np.array(images, dtype=np.float32) / 255.0
    labels = np.array(labels, dtype=np.int32)
    return images, labels


X, y = load_dataset()

print(f"Dataset shape: X={X.shape}, y={y.shape}")
print(f"  Free samples:   {np.sum(y == 0)}")
print(f"  Booked samples: {np.sum(y == 1)}")
print(f"  Pixel range:    [{X.min():.2f}, {X.max():.2f}]")

In [ ]:
# One-hot encode labels for categorical_crossentropy
y_onehot = keras.utils.to_categorical(y, num_classes=2)

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X, y_onehot, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set:   X_train={X_train.shape}, y_train={y_train.shape}")
print(f"Validation set: X_val={X_val.shape},   y_val={y_val.shape}")

# Show class distribution
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for i, (split_name, split_y) in enumerate([("Train", y_train), ("Validation", y_val)]):
    counts = [np.sum(split_y[:, 0]), np.sum(split_y[:, 1])]
    axes[i].bar(CLASS_NAMES, counts, color=["#4CAF50", "#F44336"])
    axes[i].set_title(f"{split_name} Set Distribution")
    axes[i].set_ylabel("Count")
    for j, c in enumerate(counts):
        axes[i].text(j, c + 0.5, str(int(c)), ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

---
## Section 7 — Build CNN Model

We build a simple but effective CNN architecture:

```
Input (64×64×3)
  → Conv2D(32, 3×3) + ReLU
  → MaxPooling2D(2×2)
  → Conv2D(64, 3×3) + ReLU
  → MaxPooling2D(2×2)
  → Flatten
  → Dense(64, ReLU)
  → Dropout(0.3)
  → Dense(2, Softmax)
```

This is lightweight enough to train quickly on a small dataset while still capturing the visual difference between yellow (free) and non-yellow (booked) cells.

In [ ]:
def build_cnn(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=2):
    model = keras.Sequential([
        layers.Input(shape=input_shape),

        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),

        layers.Flatten(),
        layers.Dense(64, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation="softmax"),
    ])
    return model


model = build_cnn()
model.summary()

---
## Section 8 — Train the Model

We compile the model with:
- **Optimizer:** Adam (adaptive learning rate)
- **Loss:** Categorical cross-entropy (for 2-class one-hot labels)
- **Metrics:** Accuracy

Training runs for **10 epochs** with a batch size of 16. We then plot the training and validation accuracy/loss curves.

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

EPOCHS = 10
BATCH_SIZE = 16

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, y_val),
    verbose=1,
)

print(f"\nFinal training accuracy:   {history.history['accuracy'][-1]:.4f}")
print(f"Final validation accuracy: {history.history['val_accuracy'][-1]:.4f}")

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history["accuracy"], label="Train Accuracy", marker="o")
ax1.plot(history.history["val_accuracy"], label="Val Accuracy", marker="s")
ax1.set_title("Model Accuracy")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history["loss"], label="Train Loss", marker="o")
ax2.plot(history.history["val_loss"], label="Val Loss", marker="s")
ax2.set_title("Model Loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Save the trained model
model.save("slot_classifier_cnn.keras")
print("Model saved to: slot_classifier_cnn.keras")

---
## Section 9 — Predict Slot Status (Utility Functions)

We create reusable helper functions:
- `predict_slot_cell()` — classify a single cell numpy array (BGR) as FREE or BOOKED.
- `predict_slot()` — load from disk and classify.
- `extract_time_labels_ocr()` — OCR the time column to map pixel rows to clock times.

These are used in Section 10 when the user queries a specific dentist.

In [ ]:
def predict_slot_cell(cell_bgr, model, img_size=IMG_SIZE):
    """Classify a single slot cell (BGR numpy array) as FREE or BOOKED."""
    cell_resized = cv2.resize(cell_bgr, (img_size, img_size))
    cell_rgb = cv2.cvtColor(cell_resized, cv2.COLOR_BGR2RGB)
    cell_norm = cell_rgb.astype(np.float32) / 255.0
    cell_batch = np.expand_dims(cell_norm, axis=0)

    preds = model.predict(cell_batch, verbose=0)
    class_idx = np.argmax(preds[0])
    confidence = preds[0][class_idx]
    label = "FREE" if class_idx == 0 else "BOOKED"
    return label, confidence


def predict_slot(image_path, model=model, img_size=IMG_SIZE):
    """Load a slot image from disk and classify as FREE or BOOKED."""
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Cannot load: {image_path}")
    return predict_slot_cell(img, model, img_size)


def extract_time_labels_ocr(image_bgr, time_col, content_y_top, y_bottom):
    """OCR the time column to get (y_pixel, hour) pairs."""
    region = image_bgr[content_y_top:y_bottom, time_col[0]:time_col[1]]
    gray_tc = cv2.cvtColor(region, cv2.COLOR_BGR2GRAY)
    scale = 5
    scaled = cv2.resize(gray_tc, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
    _, binarized = cv2.threshold(scaled, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    data = pytesseract.image_to_data(
        binarized,
        config="--psm 4 -c tessedit_char_whitelist=0123456789",
        output_type=pytesseract.Output.DICT,
    )

    raw = []
    for i, text in enumerate(data["text"]):
        text = text.strip()
        if not text:
            continue
        try:
            hour = int(text)
        except ValueError:
            continue
        if 6 <= hour <= 22:
            raw.append((data["top"][i] // scale + content_y_top, hour))

    raw.sort(key=lambda p: p[0])
    if not raw:
        return []
    result = [raw[0]]
    for y_px, hour in raw[1:]:
        if y_px > result[-1][0] and hour > result[-1][1]:
            result.append((y_px, hour))
    return result


def pixel_to_time(y_pixel, time_labels, start_hour=8, end_hour=20):
    """Convert a pixel y-coordinate to (hour, minute) using OCR time labels."""
    if len(time_labels) >= 2:
        y_top_lbl, h_top = time_labels[0]
        y_bot_lbl, h_bot = time_labels[-1]
    else:
        return (start_hour, 0)

    span = max(h_bot - h_top, 1)
    relative = (y_pixel - y_top_lbl) / max(y_bot_lbl - y_top_lbl, 1)
    total_min = int(h_top * 60 + relative * span * 60)
    total_min = max(start_hour * 60, min(total_min, end_hour * 60))
    return (total_min // 60, total_min % 60)


# Quick test on a few sample images
test_images = sorted(glob.glob("dataset/free/*.png"))[:3] + sorted(glob.glob("dataset/booked/*.png"))[:3]

fig, axes = plt.subplots(1, min(len(test_images), 6), figsize=(18, 4))
if len(test_images) == 1:
    axes = [axes]

for i, path in enumerate(test_images[:6]):
    label, conf = predict_slot(path)
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    color = "green" if label == "FREE" else "red"
    axes[i].imshow(img)
    axes[i].set_title(f"{label} ({conf:.1%})", color=color, fontweight="bold")
    axes[i].axis("off")

plt.suptitle("Single-Slot Predictions", fontsize=14)
plt.tight_layout()
plt.show()

---
## Section 10 — Query Availability by Dentist Name

This is the **main interactive feature**. The user enters a **dentist name**, and the system:

1. Matches the name to a detected column header (OCR'd in Section 5).
2. OCRs the time-label column to get pixel-to-time mapping.
3. Extracts each slot cell from that dentist's column.
4. Runs the CNN on each cell to classify FREE vs. BOOKED.
5. Prints a **time-labeled availability report** and a visual overlay.

**Example output:**
```
Availability for Karine Chartier:
  08:00 - 09:00  ->  BOOKED  (98.5%)
  09:00 - 10:00  ->  FREE    (97.2%)
  ...
```

In [ ]:
def match_dentist(query, dentist_names, data_cols):
    """Match a user query to a column. Supports index, exact substring, or fuzzy match."""
    # Try numeric index (1-based)
    try:
        idx = int(query) - 1
        if 0 <= idx < len(data_cols):
            return idx, data_cols[idx]
    except ValueError:
        pass

    q = query.lower().strip()

    # Substring match
    for i, name in enumerate(dentist_names):
        if not name:
            continue
        if q in name.lower() or name.lower() in q:
            return i, data_cols[i]

    # Fuzzy character-overlap fallback
    best_idx, best_score = None, 0.0
    for i, name in enumerate(dentist_names):
        if not name:
            continue
        n = name.lower()
        common = sum(1 for c in q if c in n)
        score = common / max(len(q), len(n))
        if score > best_score:
            best_score = score
            best_idx = i

    if best_idx is not None and best_score > 0.4:
        return best_idx, data_cols[best_idx]
    return None, None


# ── User Input ──────────────────────────────────────────
print("Available dentists detected from the schedule image:")
for i, name in enumerate(dentist_names):
    print(f"  {i+1}. {name if name else '(unrecognized)'}")

dentist_query = input("\nEnter dentist name (or column number): ").strip()

col_idx, col_bounds = match_dentist(dentist_query, dentist_names, data_cols)

if col_idx is None:
    print(f"\nError: '{dentist_query}' not found among detected names.")
    print("Try entering a column number instead (1, 2, 3, ...).")
else:
    matched_name = dentist_names[col_idx] or f"Column {col_idx + 1}"
    cx0, cx1 = col_bounds
    print(f"\nMatched: {matched_name}  (column {col_idx + 1}, x=[{cx0}, {cx1}])")

    # OCR time labels for pixel-to-time mapping
    time_labels_list = extract_time_labels_ocr(image_bgr, time_col, content_y_top, y_bottom)
    if time_labels_list:
        print(f"Time labels detected: {time_labels_list[0][1]}:00 to {time_labels_list[-1][1]}:00")
    else:
        print("Warning: No time labels detected; using estimated row indices.")

    # Classify each slot in this dentist's column
    margin = 3
    y = content_y_top
    row_idx = 0
    slot_results = []

    while y + row_h <= y_bottom:
        cell = image_bgr[y:y + row_h, cx0 + margin:cx1 - margin]
        if cell.size == 0:
            y += row_h
            row_idx += 1
            continue

        label, conf = predict_slot_cell(cell, model)

        time_start = pixel_to_time(y, time_labels_list) if time_labels_list else (8 + row_idx, 0)
        time_end = pixel_to_time(y + row_h, time_labels_list) if time_labels_list else (8 + row_idx + 1, 0)

        slot_results.append({
            "row": row_idx,
            "y_top": y,
            "y_bot": y + row_h,
            "time_start": time_start,
            "time_end": time_end,
            "label": label,
            "confidence": conf,
        })

        y += row_h
        row_idx += 1

    # ── Print Availability Report ──────────────────────
    print(f"\n{'='*60}")
    print(f"  Availability for: {matched_name}")
    print(f"{'='*60}")
    free_count = 0
    for s in slot_results:
        ts = f"{s['time_start'][0]:02d}:{s['time_start'][1]:02d}"
        te = f"{s['time_end'][0]:02d}:{s['time_end'][1]:02d}"
        icon = "[ FREE  ]" if s["label"] == "FREE" else "[BOOKED ]"
        color_code = "\033[92m" if s["label"] == "FREE" else "\033[91m"
        reset = "\033[0m"
        print(f"  {ts} - {te}  {color_code}{icon}{reset}  ({s['confidence']:.1%})")
        if s["label"] == "FREE":
            free_count += 1

    print(f"\n  Summary: {free_count} FREE / {len(slot_results)} total slots")
    print(f"{'='*60}")

In [ ]:
# Visual overlay — highlight the selected dentist's column
overlay = image_rgb.copy()

# Dim all other columns slightly
mask = np.ones_like(overlay, dtype=np.float32) * 0.4
mask[content_y_top:y_bottom, cx0:cx1] = 1.0
overlay = (overlay.astype(np.float32) * mask).astype(np.uint8)

# Draw slot classification rectangles on the selected column
for s in slot_results:
    if s["label"] == "FREE":
        color = (0, 255, 0)
        thickness = 3
    else:
        color = (255, 50, 50)
        thickness = 2

    cv2.rectangle(overlay, (cx0 + 2, s["y_top"]), (cx1 - 2, s["y_bot"]), color, thickness)

    ts = f"{s['time_start'][0]:02d}:{s['time_start'][1]:02d}"
    cv2.putText(overlay, f"{ts} {s['label']}", (cx0 + 5, s["y_top"] + 12),
                cv2.FONT_HERSHEY_SIMPLEX, 0.35, (255, 255, 255), 1, cv2.LINE_AA)

plt.figure(figsize=(16, 8))
plt.imshow(overlay)
plt.title(f"Availability for {matched_name} — Green=FREE, Red=BOOKED", fontsize=14)
plt.axis("off")
plt.tight_layout()
plt.show()

# Also show a bar chart summary
labels_list = [s["label"] for s in slot_results]
times_list = [f"{s['time_start'][0]:02d}:{s['time_start'][1]:02d}" for s in slot_results]
colors = ["#4CAF50" if l == "FREE" else "#F44336" for l in labels_list]

fig, ax = plt.subplots(figsize=(14, 4))
bars = ax.barh(range(len(slot_results)), [s["confidence"] for s in slot_results],
               color=colors, edgecolor="white")
ax.set_yticks(range(len(slot_results)))
ax.set_yticklabels([f"{t} - {labels_list[i]}" for i, t in enumerate(times_list)], fontsize=9)
ax.set_xlabel("CNN Confidence")
ax.set_title(f"Slot-by-Slot Confidence for {matched_name}")
ax.invert_yaxis()
ax.axvline(x=0.5, color="gray", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()

---
## Section 11 — Deeper Look: OCR Time Labels

This section provides a **closer look** at how pytesseract extracts time labels from the schedule's left-hand time column. The same logic is used internally by `extract_time_labels_ocr()` in Section 9.

Understanding this helps debug cases where time mapping is inaccurate.

In [ ]:
# time_col was already detected in Section 5 (Cell 12)
print(f"Time column: x=[{time_col[0]}, {time_col[1]}]")

# Crop and preprocess the time-label region for visualization
time_region = image_bgr[content_y_top:y_bottom, time_col[0]:time_col[1]]
time_gray = cv2.cvtColor(time_region, cv2.COLOR_BGR2GRAY)

scale = 5
time_scaled = cv2.resize(time_gray, None, fx=scale, fy=scale, interpolation=cv2.INTER_CUBIC)
_, time_binary = cv2.threshold(time_scaled, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(6, 10))
ax1.imshow(cv2.cvtColor(time_region, cv2.COLOR_BGR2RGB))
ax1.set_title("Time Column (Original)")
ax1.axis("off")
ax2.imshow(time_binary, cmap="gray")
ax2.set_title("Time Column (Binarized & Upscaled)")
ax2.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Run OCR on the time column
ocr_data = pytesseract.image_to_data(
    time_binary,
    config="--psm 4 -c tessedit_char_whitelist=0123456789",
    output_type=pytesseract.Output.DICT,
)

time_labels = []
for i, text in enumerate(ocr_data["text"]):
    text = text.strip()
    if not text:
        continue
    try:
        hour = int(text)
    except ValueError:
        continue
    if 6 <= hour <= 22:
        y_pixel = ocr_data["top"][i] // scale + content_y_top
        time_labels.append((y_pixel, hour))

# Remove duplicates and enforce monotonic increase
time_labels.sort(key=lambda p: p[0])
filtered = []
for y_px, hour in time_labels:
    if not filtered or (y_px > filtered[-1][0] and hour > filtered[-1][1]):
        filtered.append((y_px, hour))

print("Extracted Time Labels:")
print(f"{'Y-pixel':<12} {'Hour'}")
print("-" * 22)
for y_px, hour in filtered:
    print(f"{y_px:<12} {hour}:00")

if not filtered:
    print("\n⚠ No time labels detected. Try adjusting OCR parameters or image quality.")

---
## Section 12 — Final Summary

### End-to-End Pipeline
```
Schedule Image
  -> Grid Detection (HSV green border)
  -> Column OCR (pytesseract -> dentist names)
  -> Slot Extraction (crop cells per row/column)
  -> CNN Classification (FREE vs BOOKED)
  -> User queries dentist name
  -> Time-labeled availability report
```

### CNN vs. Rule-Based Approach

| Aspect | Rule-Based | CNN-Based |
|--------|-----------|----------|
| **Setup effort** | Low | Medium (label data + train) |
| **Robustness** | Brittle to theme/color changes | Generalizes from examples |
| **Accuracy on new images** | Needs constant re-tuning | Typically more stable |
| **Interpretability** | Easy to explain | Harder to inspect decisions |
| **Speed** | Very fast | Slightly slower (GPU helps) |

### Advantages of the CNN Approach
- **Learns complex visual patterns** that are hard to express as hand-crafted rules.
- **Adapts** to new scheduling software themes with just new labeled data.
- Can distinguish **more than two classes** (e.g., break, lunch, partially booked).
- Combined with **OCR column headers**, enables a natural **dentist-name query interface**.

### Limitations
- **Requires labeled data** — the initial labeling step is manual.
- **Small dataset** — with very few examples from a single image, the model may overfit. Data augmentation can help.
- **Single-image extraction** — the slot-cropping logic still relies on grid detection heuristics.
- **OCR accuracy** — practitioner names may be misread; fuzzy matching mitigates this.

### Possible Improvements
1. **Data augmentation** — flip, rotate, adjust brightness/contrast to increase effective dataset size.
2. **Transfer learning** — use a pre-trained MobileNet or ResNet backbone for better feature extraction.
3. **Semantic segmentation** — replace per-cell classification with U-Net for pixel-level free/booked masks.
4. **Multi-image training** — collect screenshots from multiple days to improve generalization.
5. **Multi-class labels** — extend to: free, booked, break, lunch, out-of-office.
6. **Web interface** — wrap the pipeline in a Flask/Streamlit app where users upload an image and select a dentist.

---

*This notebook demonstrates an end-to-end pipeline: from a raw scheduling screenshot to a trained CNN that answers "When is Dr. X available?" — designed for interview demos and educational walkthroughs.*